#### Global setups and imports

In [1]:
import os
from pathlib import Path

# Get the current directory of the notebook
notebook_path = Path.cwd()

ROOT = notebook_path.parent.parent

# Change the Working Directory for the whole process
os.chdir(ROOT)

print(f"Current Working Directory fixed to: {os.getcwd()}")

Current Working Directory fixed to: /srv/homes/onbo10/thesis_main


+ Select the GPU

In [2]:
from src.Keypoints_detection.training.utils  import *
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device= get_device()

* load general evaluation constants

In [3]:
from src.Keypoints_detection.evaluation.constants import SIGMAS, IOU_THRESHOLDS

* imports

In [4]:

import torch
from torch.utils.data import DataLoader
from src.Keypoints_detection.evaluation.eval_datasets import *
from src.Keypoints_detection.evaluation.evaluation_utils import *
import numpy as np
import pandas as pd
import yaml
from types import SimpleNamespace
from hrnet_config import cfg , update_config
from ultralytics import YOLO
from mmengine.dataset import DefaultSampler
from mmengine.runner import load_checkpoint
from mmpose.apis import init_model
from mmpose.datasets import build_dataset
from mmengine.dataset import pseudo_collate
from src.Keypoints_detection.Top_down_keypoints_detection_pipline import KeypointDetectionPipeline
from mmpose.apis import init_model
from ultralytics import YOLO
from utilities.logger import log_evaluation_results_kpts



In [5]:
os.path.exists('utils/logger.py')

False

+ Paths and data test list for HRNet

In [6]:
img_root_hrnet= 'data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_frames'
ann_root_hrnet= 'data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/extracted_bboxes_kpts'
split_path='data/SurgPose/SurgPose_for_HRNet/Extracted_left_right/video_split.yaml'
with open(split_path,'r') as f:
    splits=yaml.safe_load(f)
test_vid_list=splits['test']

+ Paths for yolov8-x-pose

In [7]:
img_root_yolo= 'data/SurgPose/SurgPose_for_YOLO/yolo_formated_surgpose_kpts/images/test'
ann_root_yolo= 'data/SurgPose/SurgPose_for_YOLO/yolo_formated_surgpose_kpts/labels/test'

* Define a path to save the results' logs

In [8]:
log_path= "results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv"

+ Names of the Valid test instances

In [9]:
# Surgpose dataset has corrupt bboxes instances, these nees do be cleaned out (method from HRNetEvaluationDataset Class)
# In order to evaluate all methods on the exact same instances we need to detect relevent keys
hrnet_dataset = HRNetEvaluationDataset(img_root_hrnet, ann_root_hrnet, mode='cropped',video_list=test_vid_list)
valid_keys = set()

for sample in hrnet_dataset.samples:
    # Create a unique key: 'video_id/left or right'
    img_name = os.path.basename(sample["img_path"])
    video_id = sample["img_path"].split('/')[-2]
    key = f"{video_id}/{img_name}_{sample['obj_id']}"
    valid_keys.add(key)

print(f"Total instances in HRNet evaluation: {len(valid_keys)}")

Total instances in HRNet evaluation: 3073


#### Evaluating pose models on cropped instances input

+ Evaluate HRNet Cropped input (without object detection predecessor)

In [10]:

# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3__cropped_out7-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment3/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth' # '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/HRNet_one_instance/Experiment1/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
model= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
device= get_device()
model.to(device)
model.eval();


In [11]:
# Load the test dataset and initiate the dataloader
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet, mode='cropped',video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

=> In this dataset class we are skipping instances where the GT bounding boxes are corrupt (H or W are < 20 px or the area < 400 , or GT kpts not in bbox)

In [12]:

num_images, precision, recall, map50, map50_95 = evaluate_HRNET_cropped(model, test_loader,device ,SIGMAS, IOU_THRESHOLDS)

num_valid= len(valid_keys)
results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("HRNet Cropped", model_weights, results, log_path= log_path)


HRNET CROPPED TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.9997
Recall         : 0.9997
mAP50          : 0.9950
mAP50-95       : 0.9401
Weights_Path   : results/Keypoints_detection/training_results/HRNet_trainings/Experiment3/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* Evaluate ViTPose (without object detection predecessor)

In [13]:


config_file = 'configs/Vitpose/vitpose_surg_7kpt.py'
checkpoint_file = 'results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth'

#Initialize the finetuned model
model = init_model(config_file, checkpoint_file, device=device)
model.eval()
dataset = build_dataset(model.cfg.test_dataloader.dataset)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    collate_fn=pseudo_collate
)


Loads checkpoint by local backend from path: results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [14]:
num_images, precision, recall, map50, map50_95 = evaluate_ViTPose_custom(model,dataloader,device,SIGMAS,IOU_THRESHOLDS)

num_valid= len(valid_keys)
results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("ViTpose-B-simple", checkpoint_file, results, log_path= log_path)



VITPOSE-B-SIMPLE TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.9997
Recall         : 0.9997
mAP50          : 0.9950
mAP50-95       : 0.9923
Weights_Path   : results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


#### Evaluating models on full images input

* Evaluate YOLOv8x-pose

In [15]:

model_weights='results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Pose_Experiment1/weights/last.pt'
yolo_model = YOLO(model_weights)
dataset = YoloPoseEvaluationDataset(img_dir=img_root_yolo, label_dir=ann_root_yolo)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)


In [16]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_YOLO(yolo_model, dataloader, device,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet)
yolo_results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("YOLOv8x-pose", model_weights, yolo_results, log_path= log_path)



YOLOV8X-POSE TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.8273
Recall         : 0.9945
mAP50          : 0.8701
mAP50-95       : 0.6614
Weights_Path   : results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Pose_Experiment1/weights/last.pt
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* Evaluate YOLOv8x-pose trained with Adam optimizer, Lr= 1e-3


In [17]:
model_weights='results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Pose_Experiment2/weights/last.pt'
yolo_model = YOLO(model_weights)
dataset = YoloPoseEvaluationDataset(img_dir=img_root_yolo, label_dir=ann_root_yolo)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [18]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_YOLO(yolo_model, dataloader, device,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet)
yolo_results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("YOLOv8x-pose(Adam)", model_weights, yolo_results, log_path= log_path)



YOLOV8X-POSE(ADAM) TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.8248
Recall         : 0.9951
mAP50          : 0.8789
mAP50-95       : 0.6833
Weights_Path   : results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Pose_Experiment2/weights/last.pt
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* HRNET full image 256*192 with data augmentation

In [19]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment1/training_checkpoints2026-02-18_14-32-56/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
hrnet_model.to(device)
hrnet_model.eval();

In [20]:
# Load the test dataset and initiate the dataloader

input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size,mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [21]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h, h_m=H_h)


results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("HRNET full image 256*192 with aug", model_weights, results, log_path= log_path)


HRNET FULL IMAGE 256*192 WITH AUG TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.8943
Recall         : 0.9343
mAP50          : 0.9129
mAP50-95       : 0.3865
Weights_Path   : results/Keypoints_detection/training_results/HRNet_trainings/Experiment1/training_checkpoints2026-02-18_14-32-56/model_epoch200.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* HRNET full image 256*192 without data augmentation

In [22]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3_out14-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment1/training_checkpoints2026-02-12_10-49-28/model_epoch200.pth'
args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg,model_weights, finetuned=True)
hrnet_model.to(device)
hrnet_model.eval();

In [23]:
# Load the test dataset and initiate the dataloader

input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]

dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size, mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [24]:

num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h, h_m=H_h)

results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("HRNET full image 256*192 without aug", model_weights, results, log_path= log_path)


HRNET FULL IMAGE 256*192 WITHOUT AUG TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.9059
Recall         : 0.9300
mAP50          : 0.9031
mAP50-95       : 0.3872
Weights_Path   : results/Keypoints_detection/training_results/HRNet_trainings/Experiment1/training_checkpoints2026-02-12_10-49-28/model_epoch200.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* HRNET full image 704*512 with data augmentation

In [25]:
# Load finetuned HRNet
cfg_file = 'configs/HRNet/w32_704x512_adam_lr1e-3_out14-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment2/training_checkpoints2026-02-12_11-06-15/model_epoch200.pth'
args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
hrnet_model= load_pretrained_HRNet(cfg,model_weights, finetuned=True)
device= get_device()
hrnet_model.to(device)
hrnet_model.eval();

In [26]:
# Load the test dataset and initiate the dataloader
input_size = (cfg.MODEL.IMAGE_SIZE[1],cfg.MODEL.IMAGE_SIZE[0])
H_h, W_h= cfg.MODEL.HEATMAP_SIZE[0],cfg.MODEL.HEATMAP_SIZE[1]
dataset = HRNetEvaluationDataset(img_root_hrnet,ann_root_hrnet,input_size=input_size,mode='full', video_list=test_vid_list)
test_loader= DataLoader(dataset,batch_size=1,shuffle=False)

In [27]:
num_images, precision, recall, map50, map50_95, num_valid = evaluate_HRNet_full_image(hrnet_model, test_loader, device ,SIGMAS, IOU_THRESHOLDS, valid_keys, ann_root_hrnet, w_m=W_h , h_m=H_h)
results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}
log_evaluation_results_kpts("HRNET full image 704*512 with aug", model_weights, results, log_path= log_path)


HRNET FULL IMAGE 704*512 WITH AUG TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : 3073
Precision      : 0.9884
Recall         : 0.9997
mAP50          : 0.9895
mAP50-95       : 0.9013
Weights_Path   : results/Keypoints_detection/training_results/HRNet_trainings/Experiment2/training_checkpoints2026-02-12_11-06-15/model_epoch200.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


#### Evaluating Top Down piplines

+ Yolov8 + ViTpose

In [28]:
det_model = YOLO("results/Keypoints_detection/training_results/YOLO_trainings/YOLO_Object_Experiment1/weights/best.pt")
det_model.eval();

In [29]:

config_file = 'configs/Vitpose/vitpose_surg_7kpt.py'
checkpoint_file = 'results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth'
model_type_vitpose='vitpose'
pose_model_vitpose = init_model(config_file, checkpoint_file, device=device)
pose_model_vitpose.eval();

Loads checkpoint by local backend from path: results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth


In [30]:

vitpose_pipeline = KeypointDetectionPipeline(det_model,pose_model_vitpose,model_type_vitpose, device= device)


In [31]:
num_images, precision, recall, map50, map50_95= evaluate_topdown_pipeline(vitpose_pipeline,img_root_yolo,ann_root_yolo, SIGMAS, IOU_THRESHOLDS,device)
num_valid = None
results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
}


100%|██████████| 2004/2004 [02:03<00:00, 16.27it/s]


In [32]:
log_evaluation_results_kpts("Pipeline: Yolov8 + ViTpose",checkpoint_file, results, log_path= log_path)


PIPELINE: YOLOV8 + VITPOSE TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : None
Precision      : 0.9920
Recall         : 0.9918
mAP50          : 0.9912
mAP50-95       : 0.9830
Weights_Path   : results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


* Yolov8 + HRNet 

In [33]:

# Load the pose model
cfg_file = 'configs/HRNet/w32_256x192_adam_lr1e-3__cropped_out7-finetune.yaml'
model_weights= 'results/Keypoints_detection/training_results/HRNet_trainings/Experiment3/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth'

args = SimpleNamespace(
        cfg=cfg_file,
        opts=[],
        modelDir='',
        logDir='',
        dataDir='',
        prevModelDir=''
    )
update_config(cfg, args)
pose_model_hrnet= load_pretrained_HRNet(cfg, model_weights, finetuned=True)
device= get_device()
pose_model_hrnet.to(device)
pose_model_hrnet.eval()
model_type_hrnet='hrnet'


In [34]:
hrnet_pipeline = KeypointDetectionPipeline(det_model,pose_model_hrnet,model_type_hrnet, device= device)

In [35]:
num_images, precision, recall, map50, map50_95= evaluate_topdown_pipeline(hrnet_pipeline,img_root_yolo,ann_root_yolo, SIGMAS, IOU_THRESHOLDS,device)
num_valid = None
results = {
    "num_images": num_images,
    "num_valid": num_valid,
    "precision": precision,
    "recall": recall,
    "map50": map50,
    "map50_95": map50_95
    }

100%|██████████| 2004/2004 [03:07<00:00, 10.66it/s]


In [36]:
log_evaluation_results_kpts("Pipeline: Yolov8 + HRNet",checkpoint_file, results, log_path= log_path)


PIPELINE: YOLOV8 + HRNET TEST EVALUATION RESULTS
--------------------------------------------------
Instances      : None
Precision      : 0.9915
Recall         : 0.9913
mAP50          : 0.9912
mAP50-95       : 0.9227
Weights_Path   : results/Keypoints_detection/training_results/ViTpose_trainings/Experiment1/best_coco_AP_epoch_160.pth
Log updated: /srv/homes/onbo10/thesis_main/results/Keypoints_detection/inference_results/metrics/keypoints_evaluation_logs.csv


#### Displaying the computed metrics

In [37]:
df = pd.read_csv(log_path)
# To see only the best results per model:
best_results = df.sort_values('mAP50-95', ascending=False).drop_duplicates('Model').drop(columns=['Timestamp', 'Images', 'Weights_Path', 'Instances'])
print(best_results)

                                  Model  Precision  Recall   mAP50  mAP50-95
1                      ViTpose-B-simple     0.9997  0.9997  0.9950    0.9923
7            Pipeline: Yolov8 + ViTpose     0.9920  0.9918  0.9912    0.9830
0                         HRNet Cropped     0.9997  0.9997  0.9950    0.9401
8              Pipeline: Yolov8 + HRNet     0.9915  0.9913  0.9912    0.9227
6     HRNET full image 704*512 with aug     0.9884  0.9997  0.9895    0.9013
3                    YOLOv8x-pose(Adam)     0.8248  0.9951  0.8789    0.6833
2                          YOLOv8x-pose     0.8273  0.9945  0.8701    0.6614
5  HRNET full image 256*192 without aug     0.9059  0.9300  0.9031    0.3872
4     HRNET full image 256*192 with aug     0.8943  0.9343  0.9129    0.3865
